# 16 — Ensemble v2 no pH (EF01): sazonal + LSTNet-12 + LGBM-nativo + DLinear-res + NNLS fit/report

Ensemble do protocolo v2 (`L=2304 → H=288`, purge/embargo ±H, 5 fatias — idêntico ao 10/12). Piso sazonal-naive-288 + 3 corretores + NNLS com **separação peso × reporte** (o ponto central do v2). 2025 intocado (benchmark futuro).

## Componentes (4, ordem do `ensemble.json`)

| # | Componente | Origem |
|---|---|---|
| 1 | `sazonal` | sazonal-naive-288 (piso, sem treino) |
| 2 | `lstnet` | **seed-mean das 5 seeds do 12** (`resultados/12-v2-lstnet-ph/modelos/lstnet_ph_s{seed}.pt` recarregados, sem re-treino; skip elegante c/ erro claro se ausentes — §8) |
| 3 | `lgbm` | **288 LGBM nativos** (1 run determinístico `seed=42`, treino no **resíduo Y−sazonal**, features §9, `booster.save_model` um `.txt` por horizonte em `modelos/lgbm_nativo/` — sem pickle, alinha com o loader Fase 2 do `app.py`) |
| 4 | `dlres` | **DLinear-res × 5 seeds** (treino no resíduo Y−sazonal, espelho do §8 do 06; **seed-mean** como componente) |

## NNLS por zona (peso × reporte)

- **Peso (fit):** fatias 1–4 (abr/jul/set/nov, 10.080 origens, stride 4 p/ o ajuste) — NNLS sobre os 4 (ou 3, sem lstnet) componentes.
- **Reporte honesto:** fatia 5 (dez/verão, 2.880 origens, **unseen p/ os pesos**, p/ a val do DLinear e p/ o treino LGBM pós-purge).
- **Reporte declarado in-sample:** fatias 1–4 cheias (os pesos viram essas origens — tabela separada, sem maquiagem).
- Caveat documentado: o componente `lstnet` do 12 usou as 5 fatias como val no treino — o reporte em dez é honesto p/ os **pesos**, não p/ o componente lstnet.

## Réguas de referência

- v1 06 ens val **0,0357** (protocolo diferente L=8640/4 fatias/sem purge — caveat, não é comparação direta).
- v2-12 lstnet val **0,0365±0,0004** (pooled 5 seeds, 5 fatias).
- v2-10 sazonal-naive val **0,0406** (piso a bater).

## Execução remota (UM job por vez — 12c/23 GB estouram com concorrência; o 13 está no remoto — NÃO rodar junto)

- **Solo (máquina livre):** `.venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/16-v2-ensemble-ph.ipynb`
- **Compartilhada:** `OMP_NUM_THREADS=4 MKL_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 .venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/16-v2-ensemble-ph.ipynb`
- Nunca `sleep` dentro do comando remoto (o canal MCP expira); `pkill -f` com o truque `[.]` (ex.: `pkill -f 'nbconvert.*16-v2-ensemble-p[h]'`).
- Estimativa (a confirmar no run): **~60–90 min solo / ~2–3 h compartilhada** — o **LGBM 288 é o gargalo** (~30k linhas × 32 feats × 150 rounds × 288 horizontes); DLinear-res × 5 seeds em 2º; resto (inferência LSTNet + tabelas/figs) ~10 min.

## Saídas (criadas pela execução, em `resultados/16-v2-ensemble-ph/`)

`metricas_por_fatia.csv` (fatia × modelo, c/ coluna `zona=fit/report`) · `metricas_zonas.csv` (pooled fit vs report) · `metricas_por_dia.csv` (45 dias-âncora, c/ `zona`) · `modelos/ensemble.json` (pesos, tracked) + `modelos/normalizacao.json` (tracked) + `modelos/lgbm_nativo/lgbm_h{000..287}.txt` + `modelos/dlinear_res_ph_s{seed}.pt` (×5, gitignored) · `figs/` 01-eda/02-limpeza/03-stl/04-forecasts/05-mae/06-val-dias/07-importancia-lgbm/08-pesos-nnls/09-zonas (espelho do 06 + pesos NNLS + comparação por zona). O `README.md` do experimento é escrito **só após a execução**, com números reais + procedência remota.

Convenções: nada in-place no 06 · nada de `src/` · nada de 2025 · sem `groupby` aninhado (pitfall pandas≥2).

In [1]:
import json
import os
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
CSV = ROOT / "dados/treino/ef01-mogi-das-cruzes_ph_2024.csv"
OUT = ROOT / "univariavel" / "resultados" / "16-v2-ensemble-ph"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)
NAT = OUT / "modelos" / "lgbm_nativo"
NAT.mkdir(parents=True, exist_ok=True)

# Protocolo v2 (travado, idêntico ao 10/12)
L, H = 2304, 288  # 8 d -> 1 d (passo 5 min)
SEASON = 288
INTERP_LIMIT = 24  # 2 h
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24"),
              ("2024-12-13", "2024-12-22")]
FIT_SLICES = VAL_SLICES[:4]  # peso NNLS: abr/jul/set/nov
REP_SLICE = VAL_SLICES[4]    # reporte honesto: dez/verão (unseen p/ os pesos)
# LSTNet1D do 12 (só p/ recarregar os .pt; tamanhos idênticos ao 02, in_ch=8)
LN, HN = 2016, 288
N_COV, N_CH = 7, 8  # tod_sin/cos + solar + f1..f4 = 7; + valor = 8
CONV_CH, CONV_K, CONV_S = 32, 12, 6
GRU_H, SKIP_H, SKIP_P = 64, 32, 48
AR_Q = 288
DROPOUT = 0.1
SEEDS = [42, 7, 123, 2024, 999]  # seeds do LSTNet-12 (seed-mean = componente)
# LGBM nativo (1 run determinístico) + DLinear-res x 5 seeds
LGB_EST, LGB_LR, LGB_LEAVES, LGB_SEED = 150, 0.05, 31, 42
LGB_STRIDE = 2  # espelho do 06
DL_SEEDS = [42, 7, 123, 2024, 999]
DL_EPOCHS, DL_PAT, DL_BATCH, DL_LR = 30, 5, 512, 1e-3  # espelho do 06
ENS_STRIDE = 4
DEVICE = torch.device("cpu")

print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)
print(f"L={L} H={H} LN={LN} fit={len(FIT_SLICES)} fatias report={REP_SLICE} seeds={SEEDS}")
print("threads:", {k: os.environ.get(k, "<unset>") for k in
      ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")},
      "| cpu:", os.cpu_count())
print("OUT:", OUT)

ROOT: /home/marcos/temporal-model | CSV existe: True | torch: 2.14.0+cpu
L=2304 H=288 LN=2016 fit=4 fatias report=('2024-12-13', '2024-12-22') seeds=[42, 7, 123, 2024, 999]
threads: {'OMP_NUM_THREADS': '<unset>', 'MKL_NUM_THREADS': '<unset>', 'OPENBLAS_NUM_THREADS': '<unset>'} | cpu: 12
OUT: /home/marcos/temporal-model/resultados/16-v2-ensemble-ph


## 1. Carga

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "pH": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

(105121, 2) 2024-01-01 00:00:00 → 2024-12-31 00:00:00
faltantes: 11665 (11.1%)


,ds,y
count,105121,93456.00000
mean,2024-07-01 12:00:00,5.90729
min,2024-01-01 00:00:00,5.21000
25%,2024-04-01 06:00:00,5.71000
50%,2024-07-01 12:00:00,5.92000
75%,2024-09-30 18:00:00,6.09000
max,2024-12-31 00:00:00,6.68000
std,NaN,0.30356


## 2. EDA

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.3)
ax[0].set_title("ph EF01 2024 — série completa (treino)")
ax[0].set_ylabel("ph")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("Ciclo diário")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva")

maior gap: 4971 passos = 414.2 h | gaps > 24 passos: 9


fig salva


## 3. Limpeza

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
gi = np.where(s.isna().to_numpy())[0]
blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
for g in blocos:
    print(f"  outage {s.index[g[0]]} → {s.index[g[-1]]} ({len(g)} slots = {len(g)*5/60:.1f} h)")

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 09–16/09")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

slots na grade: 105121 | linhas no CSV: 105121
NaN após interpolação (limite 24): 6408
  outage 2024-01-16 11:35:00 → 2024-01-18 17:50:00 (652 slots = 54.3 h)
  outage 2024-02-13 17:00:00 → 2024-02-13 17:05:00 (2 slots = 0.2 h)
  outage 2024-03-11 10:55:00 → 2024-03-11 11:55:00 (13 slots = 1.1 h)
  outage 2024-04-29 11:30:00 → 2024-05-02 03:10:00 (765 slots = 63.8 h)
  outage 2024-05-27 12:00:00 → 2024-06-13 16:10:00 (4947 slots = 412.2 h)
  outage 2024-10-19 06:05:00 → 2024-10-19 06:30:00 (6 slots = 0.5 h)
  outage 2024-10-19 13:35:00 → 2024-10-19 13:50:00 (4 slots = 0.3 h)
  outage 2024-11-25 15:15:00 → 2024-11-25 15:20:00 (2 slots = 0.2 h)
  outage 2024-12-03 13:30:00 → 2024-12-03 14:50:00 (17 slots = 1.4 h)
fig salva


## 4. ADF + STL (trecho limpo jul–ago)

In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
stat, pval, *_ = adfuller(trecho.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(trecho.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-1.79 p-valor=0.384 → NÃO estacionária


fig salva


## 5. Janelamento (L=2304) + val em 5 fatias + purge/embargo ±H + zonas peso×reporte

Idêntico ao 10/12: janelas por data de **fim**, descarte com NaN pós-interp, treino = janelas válidas fora da val que **sobrevivem ao purge** (alvo `[fim−H, fim]` sem interseção com qualquer fatia estendida `±H`). Funções `purge_train`/`signed_gap_steps` **verbatim** do 10/12 (trava por `assert`). Esperado pH: treino ~59.349 + val 12.960 ([2880, 2880, 2880, 1440, 2880]) + purge ~3.311.

Novidade do 16: a val é partida em **zona fit** (fatias 1–4: abr/jul/set/nov = 10.080 origens, onde o NNLS ajusta os pesos) e **zona report** (fatia 5: dez/verão = 2.880 origens, unseen p/ os pesos = reporte honesto).

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
n_desc_nan = int((~ok).sum())
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
ed = ends.date

# --- val por data de fim (5 fatias, incl. dez) ---
is_val = np.zeros(len(ends), dtype=bool)
per_slice = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m = (ed >= d0) & (ed <= d1)
    is_val |= m
    per_slice.append(int(m.sum()))
    print(f"fatia {a} → {b}: {int(m.sum())} janelas válidas")


def purge_train(ends, is_val, slices):
    """Descarta treino cujo alvo [fim-H, fim] intersecte fatia estendida ±H.
    Verbatim de /tmp/v2split/validate_split.py (H global)."""
    keep = is_val.copy()
    drop = np.zeros(len(ends), dtype=bool)
    for a, b in slices:
        A = pd.Timestamp(a) - pd.Timedelta(minutes=5 * H)          # ini-H
        B = pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5) \
            + pd.Timedelta(minutes=5 * H)                           # fim+H
        tgt0 = ends - pd.Timedelta(minutes=5 * H)
        hit = (~is_val) & (ends >= A) & (tgt0 <= B)  # alvo ∩ [A,B] ≠ ∅
        drop |= hit
    keep[~is_val & ~drop] = True  # treino sobrevivente
    return keep, drop  # keep=True → val ou treino válido


def signed_gap_steps(ends_tr, slices):
    """Distância (passos 5min) do alvo [fim-H,fim] à fatia mais próxima; <0 = overlap.
    Verbatim de /tmp/v2split/validate_split.py."""
    if not len(ends_tr):
        return None
    e = ends_tr.values.astype("datetime64[m]").astype(np.int64)  # min
    best = np.full(len(e), 10 ** 12)
    for a, b in slices:
        A = (pd.Timestamp(a).to_datetime64().astype("datetime64[m]").astype(int))
        B = ((pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5))
             .to_datetime64().astype("datetime64[m]").astype(int))
        s0 = e - H * 5
        ov = (e >= A) & (s0 <= B)
        gap = np.where(e < A, (A - e) // 5, np.where(s0 > B, (s0 - B) // 5, -(np.minimum(e, B) - np.maximum(s0, A)) // 5 - 1))
        best = np.minimum(best, gap)
    return best


keep, drop = purge_train(ends, is_val, VAL_SLICES)
va = np.where(is_val)[0]
tr = np.where(keep & ~is_val)[0]
print(f"treino (pós-purge): {len(tr)} | val: {len(va)} | "
      f"descartadas (NaN): {n_desc_nan} | purge: {int(drop.sum())}")

# --- asserts de cobertura por fatia (pH: fatias limpas → val cheia) ---
esperado = [288 * ((pd.Timestamp(b) - pd.Timestamp(a)).days + 1) for a, b in VAL_SLICES]
assert per_slice == esperado, f"cobertura por fatia fora do esperado: {per_slice} vs {esperado}"
assert len(va) == sum(esperado) == 12960, f"val total inesperada: {len(va)}"
assert len(va) > 1000, "val pequena demais — reposicionar fatias!"
assert int(drop.sum()) > 0, "purge removeu zero janelas — lógica inativa?"

# --- trava do purge: gap mín ≥ H+1 e overlap zero ---
gaps = signed_gap_steps(ends[tr], VAL_SLICES)
print(f"gap mín alvo-treino→val: +{int(gaps.min())} passos (exigido ≥ {H + 1}); "
      f"treino c/ alvo∩val: {int((gaps < 0).sum())}")
assert int((gaps < 0).sum()) == 0, "LEAKAGE: há alvo de treino dentro da val!"
assert int(gaps.min()) >= H + 1, f"purge/embargo falhou: gap {int(gaps.min())} < {H + 1}"

# --- dias-âncora (23:55) na val: 45 = 10+10+10+5+10 ---
daily_mask = (ends.time == pd.Timestamp("23:55").time()) & is_val
daily_idx = np.where(daily_mask)[0]
por_dia_ct = [int(((ends[daily_idx].date >= pd.Timestamp(a).date()) &
                   (ends[daily_idx].date <= pd.Timestamp(b).date())).sum())
              for a, b in VAL_SLICES]
print("dias-âncora na val:", len(daily_idx), "por fatia:", por_dia_ct)
assert len(daily_idx) == 45, f"dias-âncora inesperados: {len(daily_idx)}"
assert por_dia_ct == [10, 10, 10, 5, 10], f"âncoras por fatia: {por_dia_ct}"

# --- separação peso x reporte (ponto central do v2; posições dentro de va) ---
va_ends = ends[va]
is_rep = (va_ends.date >= pd.Timestamp(REP_SLICE[0]).date()) & \
         (va_ends.date <= pd.Timestamp(REP_SLICE[1]).date())
loc_fit, loc_rep = np.where(~is_rep)[0], np.where(is_rep)[0]
va_fit, va_rep = va[loc_fit], va[loc_rep]
print(f"zona fit (peso, fatias 1–4): {len(va_fit)} | zona report (honesto, dez): {len(va_rep)}")
assert len(va_fit) == 2880 * 3 + 1440 == 10080, len(va_fit)
assert len(va_rep) == 2880, len(va_rep)
dia_zona = np.where((ends[daily_idx].date >= pd.Timestamp(REP_SLICE[0]).date()) &
                      (ends[daily_idx].date <= pd.Timestamp(REP_SLICE[1]).date()),
                      "report", "fit")
assert (dia_zona == "report").sum() == 10 and (dia_zona == "fit").sum() == 35

fatia 2024-04-19 → 2024-04-28: 2880 janelas válidas
fatia 2024-07-20 → 2024-07-29: 2880 janelas válidas
fatia 2024-09-15 → 2024-09-24: 2880 janelas válidas
fatia 2024-11-20 → 2024-11-24: 1440 janelas válidas
fatia 2024-12-13 → 2024-12-22: 2880 janelas válidas
treino (pós-purge): 59349 | val: 12960 | descartadas (NaN): 26910 | purge: 3311
gap mín alvo-treino→val: +289 passos (exigido ≥ 289); treino c/ alvo∩val: 0
dias-âncora na val: 45 por fatia: [10, 10, 10, 5, 10]
zona fit (peso, fatias 1–4): 10080 | zona report (honesto, dez): 2880


## 6. Métricas + baselines de referência

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xtr, Ytr = X[tr], Y[tr]
Xva, Yva = X[va], Y[va]
print("treino rolante (pós-purge):")
print(pd.DataFrame({m: metricas(Ytr, p) for m, p in cheap_preds(Xtr).items()}).T.round(4).to_string())
print("val zona fit (1–4, onde o NNLS pesa):")
print(pd.DataFrame({m: metricas(Y[va_fit], p) for m, p in cheap_preds(X[va_fit]).items()}).T.round(4).to_string())
print("val zona report (dez, honesto):")
print(pd.DataFrame({m: metricas(Y[va_rep], p) for m, p in cheap_preds(X[va_rep]).items()}).T.round(4).to_string())
print("régua v2-10 sazonal val pooled 0,0406 · régua v2-12 lstnet 0,0365±0,0004 · régua v1-06 ens 0,0357 (protocolo diferente — caveat)")

treino rolante (pós-purge):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0580  0.0796  0.9967  0.9960
sazonal_naive_288  0.0551  0.0757  0.9491  0.9479
media_movel_288    0.0533  0.0715  0.9145  0.9135
val zona fit (1–4, onde o NNLS pesa):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0564  0.0809  0.9779  0.9774
sazonal_naive_288  0.0421  0.0614  0.7325  0.7320
media_movel_288    0.0468  0.0636  0.8100  0.8103
val zona report (dez, honesto):
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0721  0.0894  1.1803  1.1787
sazonal_naive_288  0.0354  0.0488  0.5815  0.5797
media_movel_288    0.0598  0.0707  0.9801  0.9783
régua v2-10 sazonal val pooled 0,0406 · régua v2-12 lstnet 0,0365±0,0004 · régua v1-06 ens 0,0357 (protocolo diferente — caveat)


## 7. Covariáveis determinísticas + canais do LSTNet (sem leakage)

Verbatim do 12: `tod_sin/cos` (como no 02) + elevação solar/90 + 4 Fourier do dia-do-ano (`f1..f4`) — futuro conhecido, sem leakage (funções de `/tmp/v2split/validate_split.py`, Mogi das Cruzes −23,52/−46,19, UTC−3). Servem aqui em dois papéis: (a) canais `Tln` p/ recarregar o LSTNet-12 no §8; (b) fonte das **5 features novas do LGBM** no §9 (4 Fourier da origem + solar do passo-alvo).

In [8]:
LAT, LON, TZ = -23.52, -46.19, -3  # Mogi das Cruzes; ts locais (UTC-3, sem DST em 2024)


def elevacao_solar(ts, lat=LAT, lon=LON, tz=TZ):
    ts = pd.DatetimeIndex(ts)
    doy = ts.dayofyear.to_numpy() + (ts.hour.to_numpy() + ts.minute.to_numpy() / 60) / 24
    g = 2 * np.pi / 365 * (doy - 1 + (ts.hour.to_numpy() - 12) / 24)
    eq = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                   - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    tst = (ts.hour.to_numpy() * 60 + ts.minute.to_numpy()) + eq + 4 * lon - 60 * tz
    ha = np.radians(tst / 4 - 180)
    cosz = np.sin(np.radians(lat)) * np.sin(decl) + np.cos(np.radians(lat)) * np.cos(decl) * np.cos(ha)
    return 90 - np.degrees(np.arccos(np.clip(cosz, -1, 1)))


def fourier_doy(ts, n=366):
    d = pd.DatetimeIndex(ts).dayofyear.to_numpy()
    return (np.sin(2 * np.pi * d / n), np.cos(2 * np.pi * d / n),
            np.sin(4 * np.pi * d / n), np.cos(4 * np.pi * d / n))


# --- canais ToD idênticos ao 02/12 + solar + fourier (todos float32) ---
TOD_SIN = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
TOD_COS = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
SOLAR = (elevacao_solar(s.index) / 90.0).astype(np.float32)  # /90 determinístico → [-1, 1]
F1, F2, F3, F4 = [a.astype(np.float32) for a in fourier_doy(s.index)]
COV = np.stack([TOD_SIN, TOD_COS, SOLAR, F1, F2, F3, F4], axis=1).astype(np.float32)
assert COV.shape == (len(s), N_COV), COV.shape
print(f"sanity solar meio-dia jan: {float(SOLAR[s.index.get_loc('2024-01-15 12:00')]):+.3f} "
      f"meia-noite: {float(SOLAR[s.index.get_loc('2024-01-15 00:00')]):+.3f}")
print(f"sanity fourier 01/jan: {[round(float(v[0]), 3) for v in (F1, F2, F3, F4)]}")

# --- janelas nativas LN=2016 (mesma construção do 02/12; p/ o LSTNet do §8) ---
val5 = s.to_numpy().astype(np.float32)
Wln = sliding_window_view(val5, LN)
Tln = sliding_window_view(COV, LN, axis=0).transpose(0, 2, 1).astype(np.float32)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - LN + 1
print(f"Wln {Wln.shape} Tln {Tln.shape} (esperado (_, {LN}) e (_, {LN}, {N_COV}))")
print(f"janelas nativas válidas: {int((rowln >= 0).sum())}/{len(ends)}")
assert Wln.shape[1] == LN and Tln.shape[1:] == (LN, N_COV), (Wln.shape, Tln.shape)
assert int((rowln >= 0).sum()) == len(ends), "cauda LN fora da grade!"
assert Tln.shape[0] == Wln.shape[0] == len(s) - LN + 1

sanity solar meio-dia jan: +0.957 meia-noite: -0.500
sanity fourier 01/jan: [0.017, 1.0, 0.034, 0.999]


Wln (103106, 2016) Tln (103106, 2016, 7) (esperado (_, 2016) e (_, 2016, 7))
janelas nativas válidas: 75620/75620


## 8. Régua LSTNet-12 (seed-mean das 5 seeds, sem re-treino)

Arquitetura `LSTNet1D` **verbatim do 12** (`in_channels=8`, mesmos tamanhos de camada) só p/ recarregar `resultados/12-v2-lstnet-ph/modelos/lstnet_ph_s{seed}.pt`. O componente é a **média das 5 seeds**. Se algum `.pt` estiver ausente (ex.: 12 ainda não publicado no Release), o notebook **segue sem o lstnet** — erro claro no log + `has_lstnet=false` no `ensemble.json` (pesos NNLS sobre os 3 restantes).

In [9]:
class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(N_CH, CONV_CH, kernel_size=CONV_K, stride=CONV_S)
        self.gru = nn.GRU(CONV_CH, GRU_H, batch_first=True)
        self.skipcell = nn.GRUCell(CONV_CH, SKIP_H)
        self.head = nn.Linear(GRU_H + SKIP_H, HN)
        self.ar = nn.Linear(AR_Q, HN)
        self.drop = nn.Dropout(DROPOUT)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, SKIP_H, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - SKIP_P] if t - SKIP_P >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -AR_Q:])
        return (yn + ya - self.beta) / g * sg + mu


CKPT12 = {sd: ROOT / "univariavel" / "resultados" / "12-v2-lstnet-ph" / "modelos" / f"lstnet_ph_s{sd}.pt"
          for sd in SEEDS}
ausentes = [str(p) for p in CKPT12.values() if not p.exists()]
modelos_ls = {}
if ausentes:
    print("ERRO — checkpoints do 12 ausentes; o ensemble segue SEM o componente lstnet:")
    for p in ausentes:
        print("  faltando:", p)
    print("para o 4º componente: execute o 12-v2-lstnet-ph ou baixe via bash scripts/baixar_modelos.sh")
    HAS_LSTNET = False
else:
    for sd in SEEDS:
        m = LSTNet1D().to(DEVICE)
        m.load_state_dict(torch.load(CKPT12[sd], map_location="cpu", weights_only=False)["state"])
        m.eval()
        modelos_ls[sd] = m
    HAS_LSTNET = True
    print("régua 12 recarregada (5 seeds):", sorted(modelos_ls))
print("HAS_LSTNET =", HAS_LSTNET)


@torch.no_grad()
def prevê_com(model_, idxs, batch=256):
    model_.eval()
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln[rowln[ii[b:b+batch]]])
        tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
        outs.append(model_(xb, tb).numpy())
    return np.concatenate(outs)


def prevê_lstnet(idxs):
    """Seed-mean das 5 seeds do 12 (componente lstnet)."""
    assert HAS_LSTNET, "lstnet ausente — veja o ERRO acima (checkpoints do 12)"
    return np.stack([prevê_com(modelos_ls[sd], idxs) for sd in SEEDS]).mean(axis=0).astype(np.float32)

régua 12 recarregada (5 seeds): [7, 42, 123, 999, 2024]
HAS_LSTNET = True


## 9. LightGBM nativo no resíduo (288 modelos, um por passo)

`base_feats` **idêntica ao 06** — lags e janelas rolantes cabem em `L=2304` (maior recuo: `lag2016` e fase `L−2016=288 ≥ 0`; `rm2016` usa a cauda de 7 d). Novidade do 16 (5 features, todas determinísticas do timestamp — sem leakage):

| # | Feature | Papel | Varia por horizonte `j`? |
|---|---|---|---|
| 1–4 | `orig_f1sin/cos`, `orig_f2sin/cos` | 4 Fourier do dia-do-ano da **origem** (`fourier_doy(ends)`) | não (estáticas por janela) |
| 5 | `solar_alvo` | elevação solar/90 do **passo-alvo j** (`ends − (H−1−j)·5min`, espelho exato de `hour_sincos`) | **sim**, como `sh/ch` |

Lista final `nomes` (32 = 25 base + 4 Fourier + `hora_sin/cos` + `solar_alvo`; ordem = ordem do `column_stack` do treino — trava por `assert`). Treino **nativo** (`lgb.train` + `booster.save_model`, um `.txt` por horizonte em `modelos/lgbm_nativo/`, 1 run determinístico `seed=42`+`deterministic` — sem pickle, mesmo formato que o loader Fase 2 do `app.py` lê via `lgb.Booster(model_file=)`). Alvo = resíduo `Y − sazonal`.

In [10]:
import lightgbm as lgb

def base_feats(Xb, E):
    # Idêntica ao 06 (cabe em L=2304 — ver texto do §9).
    cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
    phase = np.stack([Xb[:, L - 288*k] for k in range(1, 8)], axis=1)
    cols += [phase.mean(1), phase.std(1)]
    for w in [12, 36, 144, 288]:
        cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
    cols += [Xb[:, -2016:].mean(1)]
    F = np.stack(cols, axis=1)
    em = (E.hour.to_numpy()*60 + E.minute.to_numpy()).astype(np.float32)
    return F.astype(np.float32), em

def hour_sincos(em, j):
    hh = ((em - (H - 1 - j)*5) % 1440 // 60).astype(np.float32)
    return np.sin(2*np.pi*hh/24).astype(np.float32), np.cos(2*np.pi*hh/24).astype(np.float32)

def origem_fourier(E):
    """4 Fourier do dia-do-ano da origem (estáticas por janela)."""
    return np.column_stack([a.astype(np.float32) for a in fourier_doy(E)])

def solar_alvo(E, j):
    """Elevação solar/90 do passo-alvo j (varia por j, como sh/ch).
    E (=ends) é o ÚLTIMO passo-alvo; o alvo j está (H-1-j)*5min antes — espelho de hour_sincos."""
    return (elevacao_solar(E - pd.to_timedelta((H - 1 - j)*5, unit="min")) / 90.0).astype(np.float32)

nomes = ["lag1", "lag2", "lag3", "lag6", "lag12", "lag24", "lag36", "lag72", "lag144",
         "lag287", "lag288", "lag289", "lag576", "lag2016", "seasmean7", "seasstd7",
         "rm12", "rs12", "rm36", "rs36", "rm144", "rs144", "rm288", "rs288", "rm2016",
         "orig_f1sin", "orig_f1cos", "orig_f2sin", "orig_f2cos",
         "hora_sin", "hora_cos", "solar_alvo"]

tr2 = tr[::LGB_STRIDE]
Xb_tr = X[tr2]
Ends_tr = ends[tr2]
Ftr, emtr = base_feats(Xb_tr, Ends_tr)
Ff_tr = origem_fourier(Ends_tr)
Str = np.stack([Xb_tr[:, L - SEASON + h] for h in range(H)], axis=1)
Rtr = (Y[tr2] - Str).astype(np.float32)
assert Ftr.shape[1] == 25 and Ff_tr.shape[1] == 4 and len(nomes) == 32, (Ftr.shape, Ff_tr.shape)
print(f"treino lgbm: {Xb_tr.shape} → F {Ftr.shape} + fourier {Ff_tr.shape} + passo (sh/ch/solar) | resíduo std: {Rtr.std():.4f}")

params = {"objective": "regression", "metric": "l2", "num_leaves": LGB_LEAVES,
          "learning_rate": LGB_LR, "seed": LGB_SEED, "deterministic": True,
          "verbosity": -1, "force_col_wise": True}
boosters = []
t0 = time.time()
for j in range(H):
    sh, ch = hour_sincos(emtr, j)
    Xj = np.column_stack([Ftr, Ff_tr, sh, ch, solar_alvo(Ends_tr, j)])
    assert Xj.shape[1] == len(nomes) == 32, Xj.shape  # ordem = `nomes`
    bst = lgb.train(params, lgb.Dataset(Xj, label=np.ascontiguousarray(Rtr[:, j])),
                    num_boost_round=LGB_EST)
    bst.save_model(str(NAT / f"lgbm_h{j:03d}.txt"))
    boosters.append(bst)
    if (j + 1) % 48 == 0:
        print(f"  lgbm {j+1}/{H} ... {time.time()-t0:.0f}s", flush=True)
print(f"lgbm: {len(boosters)} boosters nativos em {time.time()-t0:.0f}s → {NAT} (gitignored)")
assert len(list(NAT.glob("lgbm_h*.txt"))) == H == 288

# paridade save/reload (espelho de scripts/migrar_lgbm_nativo.py)
for jj in [0, 143, 287]:
    b = lgb.Booster(model_file=str(NAT / f"lgbm_h{jj:03d}.txt"))
    sh, ch = hour_sincos(emtr[:8], jj)
    Xc = np.column_stack([Ftr[:8], Ff_tr[:8], sh, ch, solar_alvo(Ends_tr[:8], jj)])
    d = float(np.abs(boosters[jj].predict(Xc) - b.predict(Xc)).max())
    assert d <= 1e-9, (jj, d)
print("paridade nativo save/reload ok (h000/h143/h287 ≤1e-9)")

def prevê_lgbm(idxs):
    ii = np.asarray(idxs)
    Xb = X[ii]
    E = ends[ii]
    F, em = base_feats(Xb, E)
    Ff = origem_fourier(E)
    S = np.stack([Xb[:, L - SEASON + h] for h in range(H)], axis=1)
    P = np.empty((len(ii), H), dtype=np.float32)
    for j, bst in enumerate(boosters):
        sh, ch = hour_sincos(em, j)
        P[:, j] = S[:, j] + bst.predict(np.column_stack([F, Ff, sh, ch, solar_alvo(E, j)]))
    return P

imp = np.mean([b.feature_importance(importance_type="gain") for b in boosters], axis=0)
assert len(imp) == len(nomes) == 32
ordem = np.argsort(imp)[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([nomes[k] for k in ordem], imp[ordem])
ax.set_title("LightGBM nativo — importância média das features (gain, 288 horizontes)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-importancia-lgbm.png")
print("top features:", [(nomes[k], round(float(imp[k]), 1)) for k in ordem[:5]])
print("novas (fourier+solar):", [(n, round(float(imp[nomes.index(n)]), 1))
      for n in ["orig_f1sin", "orig_f1cos", "orig_f2sin", "orig_f2cos", "solar_alvo"]])
print("fig salva: 07-importancia-lgbm.png")
del Ftr, Ff_tr, Rtr, Str, Xb_tr

treino lgbm: (29675, 2304) → F (29675, 25) + fourier (29675, 4) + passo (sh/ch/solar) | resíduo std: 0.0755


  lgbm 48/288 ... 9s


  lgbm 96/288 ... 18s


  lgbm 144/288 ... 27s


  lgbm 192/288 ... 36s


  lgbm 240/288 ... 45s


  lgbm 288/288 ... 54s


lgbm: 288 boosters nativos em 54s → /home/marcos/temporal-model/resultados/16-v2-ensemble-ph/modelos/lgbm_nativo (gitignored)
paridade nativo save/reload ok (h000/h143/h287 ≤1e-9)


top features: [('rm2016', 139.1), ('rm288', 116.6), ('orig_f1cos', 95.4), ('orig_f1sin', 87.9), ('rs288', 75.1)]
novas (fourier+solar): [('orig_f1sin', 87.9), ('orig_f1cos', 95.4), ('orig_f2sin', 69.9), ('orig_f2cos', 17.7), ('solar_alvo', 7.9)]
fig salva: 07-importancia-lgbm.png


## 10. DLinear-res × 5 seeds (treino no resíduo, seed-mean como componente)

`DLinearLite` e `monta_res` **espelho do §8 do 06** (cauda `LN=2016`, alvo = resíduo `Y − sazonal`). Diferenças do 16: (a) janela v2 `L=2304` (a cauda LN é a mesma); (b) **5 seeds** `[42, 7, 123, 2024, 999]` com o mesmo treino/early-stopping do 06 por seed; (c) a val do early-stopping é a **zona fit** (`va_fit[::4]`) — dez fica unseen p/ o reporte honesto; (d) o componente é a **seed-mean** dos 5 resíduos + piso sazonal. Checkpoints `modelos/dlinear_res_ph_s{seed}.pt` (gitignored).

In [11]:
def snaive(X_):
    return np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1)

class DLinearLite(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, H)
        self.lin_s = nn.Linear(LN, H)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg

def monta_res(idxs):
    ii = np.asarray(idxs)
    return X[ii][:, -LN:].astype(np.float32), (Y[ii] - snaive(X[ii])).astype(np.float32)

Xr_tr, Rr_tr = monta_res(tr[::LGB_STRIDE])
Xr_fit, Rr_fit = monta_res(va_fit[::ENS_STRIDE])  # early-stopping SÓ na zona fit (dez unseen)
print(f"residual: treino {Xr_tr.shape} val-fit {Xr_fit.shape}")
hists_dl, bests_dl, t_dl = {}, {}, {}
t_all = time.time()
for SEED in DL_SEEDS:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    dlres = DLinearLite().to(DEVICE)
    opt = torch.optim.Adam(dlres.parameters(), lr=DL_LR)
    loss_fn = nn.MSELoss()
    tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_tr), torch.from_numpy(Rr_tr)),
                           batch_size=DL_BATCH, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_fit), torch.from_numpy(Rr_fit)),
                           batch_size=DL_BATCH)
    best, patience, hist = float("inf"), 0, []
    t0 = time.time()
    for ep in range(1, DL_EPOCHS + 1):
        dlres.train()
        for xb, yb in tr_loader:
            opt.zero_grad(); loss = loss_fn(dlres(xb), yb); loss.backward(); opt.step()
        dlres.eval(); vl = 0.0
        with torch.no_grad():
            for xb, yb in va_loader:
                vl += float(loss_fn(dlres(xb), yb)) * len(xb)
        vl /= len(va_loader.dataset)
        hist.append(vl)
        tag = ""
        if vl < best:
            best, patience, best_ep = vl, 0, ep
            torch.save({"state": dlres.state_dict(), "seed": SEED},
                       OUT / "modelos" / f"dlinear_res_ph_s{SEED}.pt")
            tag = " *"
        else:
            patience += 1
        print(f"[dlres s{SEED}] ep {ep:02d} val-fit={vl:.5f}{tag}", flush=True)
        if patience >= DL_PAT:
            break
    hists_dl[SEED], bests_dl[SEED], t_dl[SEED] = hist, (best, best_ep), time.time() - t0
    print(f"[dlres s{SEED}] em {t_dl[SEED]:.0f}s | melhor val-fit={best:.5f} (ep {best_ep})")
print(f"dlres 5 seeds em {time.time()-t_all:.0f}s")
del Xr_tr, Rr_tr, Xr_fit, Rr_fit

dlres_modelos = {}
for SEED in DL_SEEDS:
    m = DLinearLite().to(DEVICE)
    m.load_state_dict(torch.load(OUT / "modelos" / f"dlinear_res_ph_s{SEED}.pt",
                                    map_location="cpu", weights_only=False)["state"])
    m.eval()
    dlres_modelos[SEED] = m
print("dlres checkpoints:", sorted(dlres_modelos))

@torch.no_grad()
def prevê_dlres(idxs, batch=512):
    """Piso sazonal + seed-mean dos 5 resíduos (componente dlres)."""
    ii = np.asarray(idxs)
    Ps = snaive(X[ii])
    Xt = torch.from_numpy(X[ii][:, -LN:].astype(np.float32))
    Rs = []
    for m in dlres_modelos.values():
        outs = []
        for b in range(0, len(Xt), batch):
            outs.append(m(Xt[b:b+batch]).numpy())
        Rs.append(np.concatenate(outs))
    return (Ps + np.stack(Rs).mean(axis=0)).astype(np.float32)

residual: treino (29675, 2016) val-fit (2520, 2016)


[dlres s42] ep 01 val-fit=0.00343 *


[dlres s42] ep 02 val-fit=0.00327 *


[dlres s42] ep 03 val-fit=0.00320 *


[dlres s42] ep 04 val-fit=0.00314 *


...

[dlres s999] ep 23 val-fit=0.00287


[dlres s999] ep 24 val-fit=0.00287


[dlres s999] ep 25 val-fit=0.00294


[dlres s999] ep 26 val-fit=0.00292


[dlres s999] em 44s | melhor val-fit=0.00284 (ep 21)
dlres 5 seeds em 213s
dlres checkpoints: [7, 42, 123, 999, 2024]


## 11. NNLS peso×reporte + `ensemble.json` + `normalizacao.json`

**Peso:** NNLS nos 4 (ou 3, sem lstnet) componentes sobre a zona fit subamostrada (`va_fit[::4]`, 2.520 origens). **Reporte:** zona report cheia (dez, 2.880 — honesto) + zonas fit cheias (in-sample declarado). Salva `modelos/ensemble.json` (`{pesos}`, tracked) + `modelos/normalizacao.json` (tracked).

In [12]:
from scipy.optimize import nnls

COMPS = ["sazonal"] + (["lstnet"] if HAS_LSTNET else []) + ["lgbm", "dlres"]
print("componentes do NNLS:", COMPS)

t0 = time.time()
P_va = {"sazonal": snaive(Xva).astype(np.float32)}
if HAS_LSTNET:
    P_va["lstnet"] = prevê_lstnet(va)
P_va["lgbm"] = prevê_lgbm(va)
P_va["dlres"] = prevê_dlres(va)
print(f"inferência val cheia ({len(va)} origens) em {time.time()-t0:.0f}s | shapes:",
      {k: v.shape for k, v in P_va.items()})

# --- NNLS SÓ na zona fit (subamostrada; dez nunca entra no ajuste) ---
loc_nnls = loc_fit[::ENS_STRIDE]
A = np.column_stack([P_va[c][loc_nnls].ravel() for c in COMPS])
w, _ = nnls(A, Yva[loc_nnls].ravel())
pesos = {c: round(float(v), 4) for c, v in zip(COMPS, w)}
for c in ["sazonal", "lstnet", "lgbm", "dlres"]:  # chaves estáveis (0.0 = ausente)
    pesos.setdefault(c, 0.0)
print("pesos ensemble (nnls na zona fit):", pesos)

json.dump({"pesos": pesos, "mode": "nnls-ensemble sobre " + "+".join(COMPS),
           "val_slices": VAL_SLICES, "fit_slices": FIT_SLICES, "report_slice": REP_SLICE,
           "components": COMPS, "has_lstnet": HAS_LSTNET},
          open(OUT / "modelos" / "ensemble.json", "w"))
json.dump({"mode": "sazonal-naive + lstnet-12-seedmean + lgbm-nativo-288 + dlinear-res-seedmean + nnls-fit/report",
           "L": L, "H": H, "LN": LN, "lgbm_features": nomes,
           "lgbm_native_dir": "modelos/lgbm_nativo", "lgbm_glob": "lgbm_h{j:03d}.txt",
           "val_slices": VAL_SLICES, "fit_slices": FIT_SLICES, "report_slice": REP_SLICE,
           "components": COMPS, "seeds_lstnet": SEEDS if HAS_LSTNET else [],
           "seeds_dlres": DL_SEEDS, "has_lstnet": HAS_LSTNET},
          open(OUT / "modelos" / "normalizacao.json", "w"))
print("ensemble.json + normalizacao.json salvos (tracked)")

def ensemble(Pdict):
    return sum(pesos[c] * Pdict[c] for c in COMPS)

En_va = ensemble(P_va)
P_d = {"sazonal": snaive(X[daily_idx]).astype(np.float32)}
if HAS_LSTNET:
    P_d["lstnet"] = prevê_lstnet(daily_idx)
P_d["lgbm"] = prevê_lgbm(daily_idx)
P_d["dlres"] = prevê_dlres(daily_idx)
En_d = ensemble(P_d)
print("ENS zona fit (in-sample declarado):", {k: round(v, 4) for k, v in metricas(Yva[loc_fit], En_va[loc_fit]).items()})
print("ENS zona report (honesto, dez):   ", {k: round(v, 4) for k, v in metricas(Yva[loc_rep], En_va[loc_rep]).items()})

componentes do NNLS: ['sazonal', 'lstnet', 'lgbm', 'dlres']


inferência val cheia (12960 origens) em 38s | shapes: {'sazonal': (12960, 288), 'lstnet': (12960, 288), 'lgbm': (12960, 288), 'dlres': (12960, 288)}
pesos ensemble (nnls na zona fit): {'sazonal': 0.0694, 'lstnet': 0.7048, 'lgbm': 0.0889, 'dlres': 0.1366}
ensemble.json + normalizacao.json salvos (tracked)


ENS zona fit (in-sample declarado): {'MAE': 0.0346, 'RMSE': 0.0494, 'MAPE': 0.6011, 'sMAPE': 0.6012}
ENS zona report (honesto, dez):    {'MAE': 0.0324, 'RMSE': 0.0417, 'MAPE': 0.5305, 'sMAPE': 0.5304}


## 12. Tabelas (por fatia c/ zona, pooled por zona, dias-âncora c/ zona)

Sem `groupby` aninhado (pitfall pandas≥2): só máscaras booleanas + laços.

In [13]:
MODELOS_TAB = ["sazonal"] + (["lstnet"] if HAS_LSTNET else []) + ["lgbm", "dlres", "ens"]
PRED_VA = dict(P_va); PRED_VA["ens"] = En_va
PRED_D = dict(P_d); PRED_D["ens"] = En_d

# --- por fatia: 5 fatias x modelos (zona=fit p/ 1–4, report p/ dez) ---
rows_f = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m_loc = (va_ends.date >= d0) & (va_ends.date <= d1)
    zona = "report" if (a, b) == tuple(REP_SLICE) else "fit"
    for mdl in MODELOS_TAB:
        mm = metricas(Yva[m_loc], PRED_VA[mdl][m_loc])
        rows_f.append({"fatia": f"{a}→{b}", "zona": zona, "modelo": mdl, **mm})
tab_f = pd.DataFrame(rows_f, columns=["fatia", "zona", "modelo", "MAE", "RMSE", "MAPE", "sMAPE"]).round(4)
tab_f.to_csv(OUT / "metricas_por_fatia.csv", index=False)
assert len(tab_f) == 5 * len(MODELOS_TAB) and (tab_f["zona"] == "report").sum() == len(MODELOS_TAB)
print("=== val por fatia (MAE) ===")
print(tab_f.pivot(index="fatia", columns="modelo", values="MAE").to_string())

# --- pooled por zona: fit (in-sample declarado) vs report (honesto) ---
rows_z = []
for zona, loc in [("fit", loc_fit), ("report", loc_rep)]:
    for mdl in MODELOS_TAB:
        mm = metricas(Yva[loc], PRED_VA[mdl][loc])
        rows_z.append({"zona": zona, "modelo": mdl, **mm})
tab_z = pd.DataFrame(rows_z, columns=["zona", "modelo", "MAE", "RMSE", "MAPE", "sMAPE"]).round(4)
tab_z.to_csv(OUT / "metricas_zonas.csv", index=False)
print("=== val pooled por zona ===")
print(tab_z.to_string(index=False))

# --- dias-âncora (45; MAE por dia e modelo + zona) ---
Yd = Y[daily_idx]
datas = [str(ends[i].date()) for i in daily_idx]
fatias_d = []
for dt_ in ends[daily_idx].date:
    for a, b in VAL_SLICES:
        if pd.Timestamp(a).date() <= dt_ <= pd.Timestamp(b).date():
            fatias_d.append(f"{a}→{b}"); break
por_dia = pd.DataFrame({"fatia": fatias_d, "zona": dia_zona}, index=datas)
for mdl in MODELOS_TAB:
    por_dia[mdl] = [mae(Yd[k:k+1], PRED_D[mdl][k:k+1]) for k in range(len(Yd))]
por_dia.to_csv(OUT / "metricas_por_dia.csv")
assert len(por_dia) == 45 and (por_dia["zona"] == "report").sum() == 10
print("=== dias-âncora (45) ===")
print(por_dia.round(4).to_string())
print(f"\nMelhor na zona report: {tab_z[tab_z['zona'] == 'report'].sort_values('MAE').iloc[0][['modelo', 'MAE']].to_dict()}")
print(f"pesos ensemble: {pesos}")

=== val por fatia (MAE) ===
modelo                  dlres     ens    lgbm  lstnet  sazonal
fatia                                                         
2024-04-19→2024-04-28  0.0236  0.0224  0.0283  0.0229   0.0282
2024-07-20→2024-07-29  0.0349  0.0324  0.0354  0.0332   0.0391
2024-09-15→2024-09-24  0.0586  0.0558  0.0788  0.0568   0.0693
2024-11-20→2024-11-24  0.0228  0.0210  0.0325  0.0218   0.0215
2024-12-13→2024-12-22  0.0302  0.0324  0.0354  0.0345   0.0354


=== val pooled por zona ===
  zona  modelo    MAE   RMSE   MAPE  sMAPE
   fit sazonal 0.0421 0.0614 0.7325 0.7320
   fit  lstnet 0.0354 0.0502 0.6140 0.6141
   fit    lgbm 0.0454 0.0648 0.7903 0.7878
   fit   dlres 0.0367 0.0518 0.6371 0.6372
   fit     ens 0.0346 0.0494 0.6011 0.6012
report sazonal 0.0354 0.0488 0.5815 0.5797
report  lstnet 0.0345 0.0436 0.5643 0.5643
report    lgbm 0.0354 0.0486 0.5806 0.5797
report   dlres 0.0302 0.0407 0.4970 0.4959
report     ens 0.0324 0.0417 0.5305 0.5304
=== dias-âncora (45) ===
                            fatia    zona  sazonal  lstnet    lgbm   dlres     ens
2024-04-19  2024-04-19→2024-04-28     fit   0.0262  0.0214  0.0300  0.0232  0.0219
2024-04-20  2024-04-19→2024-04-28     fit   0.0623  0.0405  0.0625  0.0450  0.0462
2024-04-21  2024-04-19→2024-04-28     fit   0.0226  0.0177  0.0222  0.0171  0.0167
2024-04-22  2024-04-19→2024-04-28     fit   0.0237  0.0200  0.0264  0.0181  0.0207
2024-04-23  2024-04-19→2024-04-28     fit   0.0327  0.0214 

## 13. Figuras (espelho do 06 + pesos NNLS + comparação por zona)

In [14]:
# --- 04-forecasts: 3 origens do treino (real x componentes x ens) ---
ks = [0, len(tr) // 2, -1]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
cp_tr = cheap_preds(X[tr])
trks = tr[ks]
Pd_tr = {"sazonal": snaive(X[trks]).astype(np.float32)}
if HAS_LSTNET:
    Pd_tr["lstnet"] = prevê_lstnet(trks)
Pd_tr["lgbm"] = prevê_lgbm(trks)
Pd_tr["dlres"] = prevê_dlres(trks)
En_tr = ensemble(Pd_tr)
for ax, k in zip(axes, ks):
    tf = pd.date_range(ends[tr[k]] - pd.Timedelta(minutes=5*(H-1)), ends[tr[k]], freq="5min")
    pos = list(ks).index(k)
    ax.plot(tf, Y[tr[k]], "k-", lw=1.5, label="real")
    ax.plot(tf, cp_tr["sazonal_naive_288"][pos], "--", lw=1, label="sazonal-naive")
    if HAS_LSTNET:
        ax.plot(tf, Pd_tr["lstnet"][pos], lw=1, alpha=0.6, label="lstnet(12-seedmean)")
    ax.plot(tf, Pd_tr["lgbm"][pos], lw=1, alpha=0.9, label="lgbm-nativo")
    ax.plot(tf, En_tr[pos], lw=1.2, alpha=0.9, label="ens")
    ax.set_title(f"origem {ends[tr[k]]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

# --- 05-mae: barras por zona (fit in-sample x report honesto) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, zona in zip(axes, ["fit", "report"]):
    tzz = tab_z[tab_z["zona"] == zona].set_index("modelo")["MAE"].sort_values()
    tzz.plot.barh(ax=ax)
    ax.set_title(f"MAE zona {zona} ({'in-sample declarado' if zona == 'fit' else 'honesto, dez'}) — menor = melhor")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

# --- 06-val-dias: MAE por dia-âncora (sombra = zona report/dez) ---
fig, ax = plt.subplots(figsize=(12, 3.5))
pdf = por_dia
for col, ls in [("sazonal", "--"), ("lstnet", "-."), ("ens", "-"), ("lgbm", ":")]:
    if col in pdf.columns:
        ax.plot(pd.to_datetime(pdf.index), pdf[col], ls, lw=1.1, label=col)
d_rep = pd.to_datetime(pdf.index[pdf["zona"] == "report"])
ax.axvspan(d_rep.min() - pd.Timedelta(hours=12), d_rep.max() + pd.Timedelta(hours=12),
            color="orange", alpha=0.15, label="zona report (dez)")
ax.set_title("ph — MAE por dia-âncora na val (zona fit + report honesto)")
ax.legend(fontsize=8); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-val-dias.png")

# --- 08-pesos-nnls: pesos do ensemble ---
fig, ax = plt.subplots(figsize=(8, 3.5))
pd.Series({c: pesos[c] for c in COMPS}).sort_values().plot.barh(ax=ax)
ax.set_title("Pesos NNLS (ajuste na zona fit 1–4; reporte honesto em dez)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "08-pesos-nnls.png")

# --- 09-zonas: cada componente, fit x report lado a lado ---
fig, ax = plt.subplots(figsize=(8, 3.5))
wdt = 0.35
xs = np.arange(len(MODELOS_TAB))
for i, zona in enumerate(["fit", "report"]):
    tzz = tab_z[tab_z["zona"] == zona].set_index("modelo").loc[MODELOS_TAB]["MAE"]
    ax.bar(xs + i * wdt, tzz.values, wdt, label=zona + (" (in-sample)" if zona == "fit" else " (honesto)"))
ax.set_xticks(xs + wdt / 2); ax.set_xticklabels(MODELOS_TAB, fontsize=8)
ax.set_title("MAE por componente: zona fit x zona report")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "09-zonas.png")
print("figs salvas")

figs salvas


## 14. Conclusões (preencher com números reais após a execução)

Réguas v2 (`metricas_zonas.csv` = primária: `report` honesto × `fit` in-sample declarado; `metricas_por_fatia.csv` mostra cada fatia c/ `zona`). Réguas de referência: **v1-06 ens 0,0357** (protocolo diferente — caveat), **v2-12 lstnet 0,0365±0,0004**, **v2-10 sazonal 0,0406** (piso). Checkpoints em `modelos/` para o benchmark futuro (`lgbm_nativo/*.txt` + `dlinear_res_ph_s*.pt` vão ao Release; só `ensemble.json`/`normalizacao.json` ficam no git).

### Protocolo v2 (resumo p/ o README do experimento)

- Janelas `L=2304 → H=288` (8 d → 1 d, 5 min), interp `time` limite 24, descarte com NaN; val 5 fatias por data de fim (19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]**); purge/embargo ±H (gap mín +289; trava por `assert`).
- Componentes: sazonal-naive-288 (piso) + LSTNet-12 seed-mean (5 `.pt` recarregados) + 288 LGBM nativos no resíduo (32 feats: 25 base do 06 + 4 Fourier da origem + hora_sin/cos + solar do passo-alvo; 1 run determinístico) + DLinear-res seed-mean (5 seeds, val do early-stopping na zona fit).
- NNLS com separação peso×reporte: pesos nas fatias 1–4, reporte honesto em dez + in-sample declarado em 1–4.

### Procedência da execução (preencher no commit da execução)

- Host remoto: `temporal-remote` 192.168.1.6 · work dir: `/home/marcos/temporal-model` · data: `2026-09-17` · threads: `solo (12c, threads unset)`
- Pós-execução: escrever `resultados/16-v2-ensemble-ph/README.md` (espelho do 06 + seção peso×reporte), indexar em `resultados/README.md` + `notebooks/README.md` + README §7 — com números reais. Não commitar `modelos/*.pt` nem `modelos/lgbm_nativo/*.txt` (vão ao Release via `scripts/baixar_modelos.sh`).